# astar_baseline

Google Colab version with the implementation embedded directly in the notebook. Plots and generated figures are displayed inline after execution.


In [ ]:
%matplotlib inline
!pip -q install numpy matplotlib pandas dimod dwave-neal ortools


In [ ]:
"""
=============================================================================
Classical A* Baseline with Occlusion Penalty — Task 3 Addition
=============================================================================
Per Dr. Ali's feedback, this script adds a classical occlusion-aware
grid planner (A*) to the Task 3 benchmark, comparing it against SA/Neal/HUBO
on the same 8x8 scenario.

WHAT THIS DOES
--------------
  1. Runs A* on the same 8x8 grid with:
       - obstacle avoidance (hard)
       - occlusion penalty (line-of-sight to target)
       - same start (0,0) and target (7,7)
       - 4-connected movement
       - planning horizon T=15
  2. Computes the same metrics as our SA/Neal comparison:
       - Best soft cost (occlusion + proximity + goal + terminal)
       - Total runtime
       - Feasibility (always 100% by construction for A*)
       - Reaches target (yes/no)
       - Visibility score (cells where target is visible)
  3. Saves trajectory plot + JSON results
  4. Generates a combined comparison table (A* vs SA vs Neal)

EXPECTED RUNTIME
----------------
  A* itself: under 1 second
  Plotting + saving: a few seconds
  TOTAL: under 10 seconds

REQUIREMENTS
------------
  - task2_grid_hubo.py in same folder (for Scenario class)
  - pip install numpy matplotlib

HOW TO RUN
----------
  python astar_baseline.py

OUTPUTS  ->  ./outputs/astar_baseline/
  astar_trajectory.png
  astar_results.json
  comparison_table.png
=============================================================================
"""

import os
import sys
import json
import time
import heapq
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Make Task 2 modules importable
TASK2_DIR = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, TASK2_DIR)

from task2_grid_hubo import Scenario, GridHUBO

OUT_DIR = os.path.join(TASK2_DIR, "outputs", "astar_baseline")
os.makedirs(OUT_DIR, exist_ok=True)


# ===========================================================================
# 1.  A* PLANNER WITH OCCLUSION PENALTY
# ===========================================================================

def heuristic(cell_a, cell_b):
    """Manhattan distance — admissible for 4-connected grid."""
    return abs(cell_a[0] - cell_b[0]) + abs(cell_a[1] - cell_b[1])


def cell_cost(cell, scenario, occlusion_lookup):
    """
    Per-cell stepping cost:
      - obstacle cells: infinite (forbidden)
      - other cells: 1.0 (base move cost) + lambda_occ * occlusion_count
                       + lambda_prox * (1 if buffer cell)
    """
    if cell in scenario.obstacles:
        return float("inf")
    base = 1.0
    occ_pen = scenario.lambda_occ * occlusion_lookup.get(cell, 0)
    v = scenario.cell_index(*cell)
    prox_pen = scenario.lambda_prox if v in scenario.buffer_indices else 0.0
    return base + occ_pen + prox_pen


def build_occlusion_lookup(scenario):
    """
    For each non-obstacle cell, count how many obstacle cells lie on the
    Bresenham line from the cell to the target.
    """
    target = scenario.target
    obstacles = set(scenario.obstacles)
    lookup = {}

    def bresenham(r0, c0, r1, c1):
        """Yields cells along the line from (r0,c0) to (r1,c1), exclusive of endpoints."""
        dr = abs(r1 - r0)
        dc = abs(c1 - c0)
        sr = 1 if r0 < r1 else -1
        sc = 1 if c0 < c1 else -1
        err = dr - dc
        r, c = r0, c0
        while (r, c) != (r1, c1):
            if (r, c) not in [(r0, c0), (r1, c1)]:
                yield (r, c)
            e2 = 2 * err
            if e2 > -dc:
                err -= dc
                r += sr
            if e2 < dr:
                err += dr
                c += sc

    gs = scenario.grid_size
    for r in range(gs):
        for c in range(gs):
            if (r, c) in obstacles:
                continue
            count = 0
            for line_cell in bresenham(r, c, target[0], target[1]):
                if line_cell in obstacles:
                    count += 1
            lookup[(r, c)] = count
    return lookup


def astar_with_occlusion(scenario, horizon=None):
    """
    Run A* from start to target, with cells weighted by obstacle + occlusion
    + proximity penalties. Returns the best path found.

    If horizon is given, the path is padded/truncated to that length.
    """
    start = scenario.start
    target = scenario.target
    occlusion_lookup = build_occlusion_lookup(scenario)

    # A* search
    # Each priority queue entry: (f_score, g_score, cell, parent_idx)
    open_heap = []
    heapq.heappush(open_heap, (heuristic(start, target), 0.0, start, None))

    came_from = {}  # cell -> (parent_cell, g_score_to_reach)
    g_scores = {start: 0.0}

    gs = scenario.grid_size
    while open_heap:
        f, g, current, parent = heapq.heappop(open_heap)

        if current in came_from and g_scores[current] < g:
            continue  # already found a better path to this cell
        came_from[current] = parent

        if current == target:
            break

        # 4-connected neighbors
        r, c = current
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < gs and 0 <= nc < gs):
                continue
            neighbor = (nr, nc)
            step_cost = cell_cost(neighbor, scenario, occlusion_lookup)
            if step_cost == float("inf"):
                continue
            new_g = g + step_cost
            if neighbor not in g_scores or new_g < g_scores[neighbor]:
                g_scores[neighbor] = new_g
                f_score = new_g + heuristic(neighbor, target)
                heapq.heappush(open_heap, (f_score, new_g, neighbor, current))

    # Reconstruct path
    if target not in came_from:
        return None  # no path found

    path = []
    cur = target
    while cur is not None:
        path.append(cur)
        cur = came_from.get(cur)
    path.reverse()

    # Pad to horizon length if needed (UAV stays at target)
    if horizon is not None and len(path) < horizon:
        path = path + [target] * (horizon - len(path))
    elif horizon is not None and len(path) > horizon:
        # Path is longer than horizon — truncate (rare; means infeasible)
        path = path[:horizon]

    return path, occlusion_lookup


# ===========================================================================
# 2.  METRIC EVALUATION (uses the same HUBO terms for fair comparison)
# ===========================================================================

def evaluate_trajectory(trajectory, scenario, grid):
    """Compute the same metrics SA/Neal use."""
    bits = np.zeros(grid.num_vars, dtype=np.int8)
    for t, cell in enumerate(trajectory):
        v = scenario.cell_index(*cell)
        bits[t * grid.V + v] = 1

    # Decompose energy using HUBO terms (same as SA/Neal)
    H_uniq = grid.H_uniq()
    H_start = grid.H_start()
    H_move = grid.H_move()
    H_obs = grid.H_obs()
    H_occ = grid.H_occ()
    H_prox = grid.H_prox()
    H_goal = grid.H_goal()
    H_terminal = grid.H_terminal()

    def w_eval(terms, weight):
        total = 0.0
        for key, coeff in terms.items():
            prod = 1
            for idx in key:
                prod *= bits[idx]
                if prod == 0: break
            total += weight * coeff * prod
        return total

    hard = (w_eval(H_uniq, scenario.lambda_uniq)
            + w_eval(H_start, scenario.lambda_start)
            + w_eval(H_move, scenario.lambda_move)
            + w_eval(H_obs, scenario.lambda_obs))
    soft = (w_eval(H_occ, scenario.lambda_occ)
            + w_eval(H_prox, scenario.lambda_prox)
            + w_eval(H_goal, scenario.lambda_goal)
            + w_eval(H_terminal, scenario.lambda_terminal))

    # Feasibility check (same as v3.1)
    feasible = True
    for t in range(grid.T):
        if sum(int(bits[t * grid.V + v]) for v in range(grid.V)) != 1:
            feasible = False
            break
    # Start
    if trajectory[0] != scenario.start:
        feasible = False
    # Adjacency
    for t in range(len(trajectory) - 1):
        if trajectory[t + 1] not in [
            (trajectory[t][0] + dr, trajectory[t][1] + dc)
            for dr, dc in [(0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)]
        ]:
            feasible = False
            break
    # Obstacle
    if any(c in scenario.obstacles for c in trajectory):
        feasible = False

    # Allow self-loop (staying in target) for path padding
    reaches_target = trajectory[-1] == scenario.target

    return {
        "hard_penalty": hard,
        "soft_cost": soft,
        "total": hard + soft,
        "feasible": feasible,
        "reaches_target": reaches_target,
    }


def compute_visibility_score(trajectory, occlusion_lookup):
    """Average visibility: fraction of trajectory cells from which target is visible."""
    visible_count = sum(1 for c in trajectory if occlusion_lookup.get(c, 0) == 0)
    return visible_count / len(trajectory)


# ===========================================================================
# 3.  PLOTTING
# ===========================================================================

def plot_astar_trajectory(scenario, trajectory, occlusion_lookup, path):
    gs = scenario.grid_size
    fig, ax = plt.subplots(figsize=(8, 8))

    # Background: obstacle + occlusion cost landscape
    img = np.zeros((gs, gs))
    for r in range(gs):
        for c in range(gs):
            if (r, c) in scenario.obstacles:
                img[r, c] = 2.0
            else:
                img[r, c] = min(occlusion_lookup.get((r, c), 0), 4) * 0.4

    cmap = plt.cm.Reds
    ax.imshow(img, cmap=cmap, origin="upper", vmin=0, vmax=2, alpha=0.65)

    # Trajectory
    ys = [c[0] for c in trajectory]
    xs = [c[1] for c in trajectory]
    ax.plot(xs, ys, "b-o", linewidth=2.5, markersize=8, zorder=5)
    for t, (y, x) in enumerate(zip(ys, xs)):
        if t % 2 == 0 or t == len(trajectory) - 1:
            ax.annotate(f"t={t}", (x, y), xytext=(5, 5),
                        textcoords="offset points", fontsize=8,
                        color="navy", fontweight="bold")

    ax.plot(scenario.start[1], scenario.start[0], "go",
            markersize=18, markeredgecolor="black", zorder=6, label="Start")
    ax.plot(scenario.target[1], scenario.target[0], "r*",
            markersize=22, markeredgecolor="black", zorder=6, label="Target")
    p_obs = mpatches.Patch(color=cmap(1.0), alpha=0.7, label="Obstacle")
    p_occ = mpatches.Patch(color=cmap(0.5), alpha=0.7, label="Occluded")
    h, l = ax.get_legend_handles_labels()
    ax.legend(handles=h + [p_obs, p_occ], loc="upper left", fontsize=9)
    ax.set_xticks(range(gs))
    ax.set_yticks(range(gs))
    ax.set_xlim(-0.5, gs - 0.5)
    ax.set_ylim(gs - 0.5, -0.5)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_title("Classical A* with Occlusion Penalty — Best Trajectory",
                 fontsize=12)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def plot_comparison_table(rows, path):
    fig, ax = plt.subplots(figsize=(12, 0.5 + 0.45 * len(rows)))
    ax.axis("off")
    tbl = ax.table(cellText=rows[1:], colLabels=rows[0],
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)
    # Header style
    for col_idx in range(len(rows[0])):
        c = tbl[(0, col_idx)]
        c.set_facecolor("#0D1B4B")
        c.set_text_props(color="white", fontweight="bold")
    # First column highlight
    for row_idx in range(1, len(rows)):
        c = tbl[(row_idx, 0)]
        c.set_text_props(fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()


# ===========================================================================
# 4.  MAIN
# ===========================================================================

def main():
    print("=" * 70)
    print("  Classical A* Baseline with Occlusion Penalty")
    print("=" * 70)
    print()
    print("Per Dr. Ali's recommendation: add an occlusion-aware classical")
    print("planner as a baseline to compare against SA/Neal/HUBO.")
    print()

    # Setup scenario with hardened penalty weights (same as v3.1)
    scenario = Scenario()
    scenario.lambda_uniq = 1000.0
    scenario.lambda_start = 1000.0
    scenario.lambda_move = 1000.0
    grid = GridHUBO(scenario)
    grid.build()

    print(f"Scenario: {scenario.grid_size}x{scenario.grid_size} grid, T={grid.T}")
    print(f"Start: {scenario.start}, Target: {scenario.target}")
    print(f"Obstacles: {len(scenario.obstacles)} cells")
    print(f"Occlusion penalty weight: λ_occ={scenario.lambda_occ}")
    print()

    # Run A*
    print("-" * 70)
    print("Running A*...")
    print("-" * 70)
    t0 = time.perf_counter()
    result = astar_with_occlusion(scenario, horizon=grid.T)
    runtime = time.perf_counter() - t0

    if result is None:
        print("  ERROR: A* found no path. Check scenario.")
        return

    trajectory, occlusion_lookup = result
    print(f"  Path length: {len(trajectory)} cells")
    print(f"  Runtime: {runtime*1000:.2f} ms")
    print(f"  Path: {trajectory}")
    print()

    # Evaluate using same HUBO metrics
    metrics = evaluate_trajectory(trajectory, scenario, grid)
    visibility = compute_visibility_score(trajectory, occlusion_lookup)

    print("-" * 70)
    print("Metrics (evaluated using same HUBO terms as SA/Neal)")
    print("-" * 70)
    print(f"  Total energy        : {metrics['total']:.2f}")
    print(f"  Hard penalty        : {metrics['hard_penalty']:.2f}")
    print(f"  Soft cost           : {metrics['soft_cost']:.2f}")
    print(f"  Feasible            : {metrics['feasible']}")
    print(f"  Reaches target      : {metrics['reaches_target']}")
    print(f"  Visibility score    : {visibility*100:.1f}%")
    print(f"                        (fraction of steps where target is visible)")
    print()

    # Save plot
    plot_path = os.path.join(OUT_DIR, "astar_trajectory.png")
    plot_astar_trajectory(scenario, trajectory, occlusion_lookup, plot_path)
    print(f"  Saved trajectory plot: {plot_path}")

    # Build comparison table  (these numbers come from v3.1 benchmark)
    # If you want fresh numbers, paste them from your v3.1 JSON
    sa_results = {
        "soft_cost": 303.43,
        "runtime_s": 47.8,
        "feasibility": 100.0,
        "reaches_target_rate": 14.0,
    }
    neal_results = {
        "soft_cost": 496.05,
        "runtime_s": 0.26,
        "feasibility": 22.0,
        "reaches_target_rate": 8.0,
    }

    # For A*, visibility computed for the path it found
    astar_visibility = visibility * 100

    table_rows = [
        ["Metric", "Classical A*", "Classical SA", "neal"],
        ["Best soft cost",
         f"{metrics['soft_cost']:.2f}",
         f"{sa_results['soft_cost']:.2f}",
         f"{neal_results['soft_cost']:.2f}"],
        ["Feasibility",
         "100%",
         f"{sa_results['feasibility']:.0f}%",
         f"{neal_results['feasibility']:.0f}%"],
        ["Reaches target",
         "Yes" if metrics['reaches_target'] else "No",
         f"{sa_results['reaches_target_rate']:.0f}% of runs",
         f"{neal_results['reaches_target_rate']:.0f}% of runs"],
        ["Visibility score",
         f"{astar_visibility:.1f}%",
         "N/A*",
         "N/A*"],
        ["Runtime per sample",
         f"{runtime*1000:.1f} ms",
         f"{sa_results['runtime_s']:.2f} s",
         f"{neal_results['runtime_s']:.2f} s"],
    ]
    table_path = os.path.join(OUT_DIR, "comparison_table.png")
    plot_comparison_table(table_rows, table_path)
    print(f"  Saved comparison table: {table_path}")
    print()
    print("  *Note: Visibility score requires per-trajectory evaluation;")
    print("   can be added to SA/Neal results similarly if desired.")

    # Save JSON
    json_payload = {
        "method": "A* with occlusion penalty",
        "scenario": {
            "grid_size": scenario.grid_size,
            "horizon_T": grid.T,
            "start": list(scenario.start),
            "target": list(scenario.target),
            "lambda_occ": scenario.lambda_occ,
        },
        "trajectory": [list(c) for c in trajectory],
        "metrics": {
            "total_energy": metrics["total"],
            "hard_penalty": metrics["hard_penalty"],
            "soft_cost": metrics["soft_cost"],
            "feasible": metrics["feasible"],
            "reaches_target": metrics["reaches_target"],
            "visibility_score": visibility,
            "runtime_ms": runtime * 1000,
        },
    }
    json_path = os.path.join(OUT_DIR, "astar_results.json")
    with open(json_path, "w") as f:
        json.dump(json_payload, f, indent=2)
    print(f"  Saved JSON: {json_path}")
    print()
    print("=" * 70)
    print("  KEY FINDING")
    print("=" * 70)
    print(f"  A* finds a feasible, target-reaching trajectory in {runtime*1000:.1f} ms")
    print(f"  with soft cost {metrics['soft_cost']:.2f} and visibility {astar_visibility:.0f}%.")
    print()
    print(f"  Comparison with quantum-inspired solvers:")
    if metrics['soft_cost'] <= sa_results['soft_cost']:
        print(f"    - A* has comparable or lower soft cost than SA in this run ({metrics['soft_cost']:.0f} vs {sa_results['soft_cost']:.0f})")
    else:
        print(f"    - SA obtained a lower best soft cost than A* ({sa_results['soft_cost']:.0f} vs {metrics['soft_cost']:.0f}), but at much higher runtime")
    print(f"    - A* is ~{sa_results['runtime_s']*1000/(runtime*1000):.0f}x faster than SA")
    print(f"    - A* is feasible by construction (100% guaranteed)")
    print()
    print("  This grounds the HUBO formulation against a conventional planner,")
    print("  confirming the problem captures sensible planning structure.")
    print("=" * 70)


if __name__ == "__main__":
    main()


## Figures produced by this notebook


In [ ]:
from pathlib import Path
from IPython.display import display, Image
pngs = sorted(Path('.').rglob('*.png'))
if not pngs:
    print('No PNG output was generated by this program.')
for p in pngs:
    print(p)
    display(Image(filename=str(p)))
